# TLS protocol classification among :443 VPN servers

**Question.** Given a VPN server that speaks TLS on port 443, can we tell whether it serves **OpenVPN-TCP**,
**SSTP**, or **both**, using only passive / cross-probe fingerprints (TLS handshake, certificate, HTTP GET
behavior, rDNS, ASN) -- *not* the protocol-specific probes the PAM 2023 authors used?

This notebook:
1. builds the per-IP feature table for U3 = `H(openvpn-tcp) ∪ H(sstp)` (641,039 servers),
2. freezes a seeded **grouped-random 70 / 20 / 10 split** into three files
   (`tls_clf_splits/train_70.csv`, `test_20.csv`, `validation_10_unseen.csv`) plus `split_metadata.json`,
   **before** any training,
3. reloads those files and reports majority / rule / gradient-boosted baselines plus feature-family ablations.

**Grouping.** Groups are *certificate SHA-256 fingerprint when present, else the /24 prefix*: servers that
reuse the same certificate stay in the same split, and certificate-less servers are kept together per /24,
so the split is random at the infrastructure level and does not leak a reused certificate across train and test.

**Leakage rules** (enforced by column choice): no protocol-probe responses (`data.sstp.result`,
`data.KeyMethod2*.result`, km1), no source-file provenance, and `in_openvpn_udp` is flagged as semi-leaky and
excluded from the baseline models.

Runtime note: full-file streams (~14 GB) -- expect a few minutes per pass.

In [1]:
import hashlib, json, re, socket, sys, time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

BASE = Path.cwd()
if not (BASE / 'hitlists').is_dir():
    BASE = Path('/home/eric-farrall/vpnecosystem_dataset')
assert (BASE / 'hitlists').is_dir(), f'run from the dataset root (cwd={BASE})'
OUT, HIT = BASE / 'output', BASE / 'hitlists'
SPLIT_DIR = BASE / 'tls_clf_splits'
SPLIT_SEED = 340
REF_DATE = datetime(2022, 8, 15, tzinfo=timezone.utc)

def stream_jsonl(path):
    with open(path, encoding='utf-8') as f:
        for line in f:
            yield json.loads(line)

def ipv4_to_int(ip):
    return int.from_bytes(socket.inet_aton(ip), 'big')

def int_to_ipv4(n):
    return socket.inet_ntoa(int(n).to_bytes(4, 'big'))

def load_ipv4_sorted(path):
    s = pd.read_csv(path, header=None, names=['ip'], dtype='string')['ip'].dropna()
    o = s.str.split('.', expand=True).astype('int64').to_numpy()
    return np.unique((o[:, 0] << 24) | (o[:, 1] << 16) | (o[:, 2] << 8) | o[:, 3])

def in_sorted(sorted_arr, values):
    idx = np.searchsorted(sorted_arr, values)
    idx = np.clip(idx, 0, len(sorted_arr) - 1)
    return sorted_arr[idx] == values

def parse_ts(txt):
    if not txt:
        return None
    try:
        return datetime.fromisoformat(txt.replace('Z', '+00:00'))
    except ValueError:
        return None

def firstval(x):
    if isinstance(x, list):
        return x[0] if x else None
    return x

IPV4_RE = re.compile(r'^\d{1,3}(\.\d{1,3}){3}$')
IPTOKEN_RE = re.compile(r'\d{1,3}[-.]\d{1,3}[-.]\d{1,3}[-.]\d{1,3}')
print('python', sys.version.split()[0], '| pandas', pd.__version__, '| base', BASE)

python 3.12.3 | pandas 3.0.5 | base /home/eric-farrall/vpnecosystem_dataset


## 1. Universe and labels

U3 is the union of the two TLS-follow-up hitlists. Every row is a VPN server on :443 detected by at least one
of the two probes; the label is derived only from hitlist membership (not from the probe payloads).

In [2]:
HIT_FILES = {
    'openvpn_tcp': HIT / 'ipv4-openvpn-tcp-hitlist.txt',
    'openvpn_udp': HIT / 'ipv4-openvpn-udp-hitlist.txt',
    'sstp':        HIT / 'ipv4-sstp-hitlist.txt',
}
H = {k: load_ipv4_sorted(v) for k, v in HIT_FILES.items()}
U3 = np.union1d(H['openvpn_tcp'], H['sstp'])
u3_set = set(U3.tolist())
in_ovpn = in_sorted(H['openvpn_tcp'], U3)
in_sstp = in_sorted(H['sstp'], U3)
in_ovpn_udp = in_sorted(H['openvpn_udp'], U3)

label = np.where(in_ovpn & in_sstp, 'both', np.where(in_ovpn, 'openvpn', 'sstp'))
counts = pd.Series(label).value_counts().rename('rows').to_frame()
counts['share_pct'] = (100 * counts['rows'] / len(U3)).round(2)
print(f'U3 = |H(openvpn_tcp) ∪ H(sstp)| = {len(U3):,}')
display(counts)

U3 = |H(openvpn_tcp) ∪ H(sstp)| = 641,039


,rows,share_pct
openvpn,453825,70.80
sstp,158083,24.66
both,29131,4.54


## 2. Feature construction

Sources and precedence (first file wins per IP; all joins are by IP only):

| family | source files | leak-safe? |
|---|---|---|
| TLS handshake + certificate | the two `-tls.txt` vulnerability scans, **Heartbleed** entry | yes: neutral handshake, not the protocol probe |
| HTTP GET over TLS / plain | the four `-https.txt` / `-http.txt` files | yes: separate generic probe |
| rDNS / domains | `output/ipv4/rDNS/ipv4-all.shuf.rdns.noerror.out` | yes: passive DNS |
| ASN / prefix | `hitlists/ipasn_ipv4_20220725.2200.dat`, longest-prefix sweep | yes: routing snapshot |
| `in_openvpn_udp` | UDP hitlist membership | **semi-leaky** (same protocol family, another scan) -- kept in files, excluded from models |

In [3]:
TLS_KEYS = [
    'handshake_present', 'tls_version', 'tls_cipher', 'tls_secure_renegotiation', 'tls_ticket',
    'tls_ocsp_stapling', 'tls_heartbeat_ext', 'tls_extended_master_secret', 'cke_rsa_present',
    'cke_rsa_params_len', 'cert_present', 'cert_validity_days', 'cert_age_days', 'cert_expired',
    'cert_self_signed', 'cert_sig_valid', 'cert_sig_algorithm', 'cert_key_algorithm', 'cert_key_bits',
    'cert_key_curve', 'cert_san_dns_count', 'cert_names_count', 'cert_has_wildcard', 'cert_cn_is_ipv4',
    'cert_cn_has_softether', 'cert_cn_localhost', 'cert_subject_eq_issuer', 'cert_is_ca',
    'cert_key_encipherment', 'cert_digital_signature', 'cert_server_auth', 'cert_validation_level',
    'cert_subject_cn', 'cert_subject_org', 'cert_issuer_cn', 'cert_issuer_org',
]

def flatten_tls(entry):
    '''Return (features dict, fingerprint or None) for one TLS-scan row.'''
    feats = dict.fromkeys(TLS_KEYS)
    d = entry.get('data') or {}
    hb = d.get('Heartbleed') if isinstance(d.get('Heartbleed'), dict) else {}
    res = hb.get('result') if isinstance(hb.get('result'), dict) else None
    hlog = (res or {}).get('handshake_log')
    feats['handshake_present'] = hlog is not None
    if not hlog:
        return feats, None
    sh = hlog.get('server_hello') or {}
    feats['tls_version'] = (sh.get('version') or {}).get('name')
    feats['tls_cipher'] = (sh.get('cipher_suite') or {}).get('name')
    feats['tls_secure_renegotiation'] = sh.get('secure_renegotiation')
    feats['tls_ticket'] = sh.get('ticket')
    feats['tls_ocsp_stapling'] = sh.get('ocsp_stapling')
    feats['tls_heartbeat_ext'] = sh.get('heartbeat')
    feats['tls_extended_master_secret'] = sh.get('extended_master_secret')
    cke = hlog.get('client_key_exchange') or {}
    rp = cke.get('rsa_params') if isinstance(cke.get('rsa_params'), dict) else None
    feats['cke_rsa_present'] = rp is not None
    feats['cke_rsa_params_len'] = rp.get('length') if rp else None
    sc = hlog.get('server_certificates') or {}
    parsed = ((sc.get('certificate') or {}).get('parsed')) or None
    feats['cert_present'] = parsed is not None
    if not parsed:
        return feats, None
    fp = parsed.get('fingerprint_sha256')
    val = parsed.get('validity') or {}
    start, end = parse_ts(val.get('start')), parse_ts(val.get('end'))
    if start and end:
        feats['cert_validity_days'] = (end - start).days
        feats['cert_age_days'] = (REF_DATE - start).days
        feats['cert_expired'] = end < REF_DATE
    sig = parsed.get('signature') or {}
    feats['cert_self_signed'] = sig.get('self_signed')
    feats['cert_sig_valid'] = sig.get('valid')
    feats['cert_sig_algorithm'] = (parsed.get('signature_algorithm') or {}).get('name')
    ski = parsed.get('subject_key_info') or {}
    feats['cert_key_algorithm'] = (ski.get('key_algorithm') or {}).get('name')
    feats['cert_key_bits'] = (ski.get('rsa_public_key') or {}).get('length')
    feats['cert_key_curve'] = (ski.get('ecdsa_public_key') or {}).get('curve')
    sub, iss = parsed.get('subject') or {}, parsed.get('issuer') or {}
    cn = firstval(sub.get('common_name'))
    feats['cert_subject_cn'] = cn
    feats['cert_subject_org'] = firstval(sub.get('organization'))
    feats['cert_issuer_cn'] = firstval(iss.get('common_name'))
    feats['cert_issuer_org'] = firstval(iss.get('organization'))
    feats['cert_subject_eq_issuer'] = (sub == iss)
    feats['cert_cn_is_ipv4'] = bool(cn and IPV4_RE.match(cn))
    feats['cert_cn_has_softether'] = bool(cn and 'softether' in cn.lower())
    feats['cert_cn_localhost'] = bool(cn and cn.lower() in ('localhost', 'user.local'))
    names = parsed.get('names') or []
    feats['cert_names_count'] = len(names)
    feats['cert_has_wildcard'] = any('*' in str(n) for n in names)
    exts = parsed.get('extensions') or {}
    feats['cert_san_dns_count'] = len((exts.get('subject_alt_name') or {}).get('dns_names') or [])
    feats['cert_is_ca'] = (exts.get('basic_constraints') or {}).get('is_ca')
    ku = exts.get('key_usage') or {}
    feats['cert_key_encipherment'] = ku.get('key_encipherment')
    feats['cert_digital_signature'] = ku.get('digital_signature')
    feats['cert_server_auth'] = (exts.get('extended_key_usage') or {}).get('server_auth')
    feats['cert_validation_level'] = parsed.get('validation_level')
    return feats, fp

t0 = time.time()
tls_store, fp_by_ip, tls_source = {}, {}, {}
for tag, path in [('openvpn', OUT / 'ipv4/OpenVPN/ipv4-openvpn-tcp.vpn.2022-08-01-1935-tls.txt'),
                  ('sstp',    OUT / 'ipv4/SSTP/ipv4-sstp.vpn.2022-08-09-1007-tls.txt')]:
    n = 0
    for e in stream_jsonl(path):
        ipi = ipv4_to_int(e['ip'])
        if ipi not in u3_set or ipi in tls_store:
            continue
        feats, fp = flatten_tls(e)
        tls_store[ipi] = feats
        if fp:
            fp_by_ip[ipi] = fp
        tls_source[ipi] = tag
        n += 1
    print(f'{tag}: kept {n:,} U3 rows')
print(f'TLS pass {time.time() - t0:.1f}s | rows with data: {len(tls_store):,}')

openvpn: kept 482,956 U3 rows


sstp: kept 158,083 U3 rows
TLS pass 31.7s | rows with data: 641,039


In [4]:
HTTPS_KEYS = ['get_https_probe_status', 'get_https_response', 'get_https_status_code',
              'get_https_status_class', 'get_https_server', 'get_https_content_type',
              'get_https_body_len', 'get_https_header_count']
HTTP_KEYS = ['get_http_probe_status', 'get_http_response', 'get_http_status_code', 'get_http_server']

def flatten_get(entry, keys_prefix):
    out = dict.fromkeys(HTTPS_KEYS if keys_prefix == 'get_https' else HTTP_KEYS)
    probe = (entry.get('data') or {}).get('http') or {}
    out[keys_prefix + '_probe_status'] = probe.get('status')
    res = probe.get('result') if isinstance(probe.get('result'), dict) else {}
    resp = res.get('response') if isinstance(res.get('response'), dict) else None
    out[keys_prefix + '_response'] = resp is not None
    if not resp:
        return out
    code = resp.get('status_code')
    if keys_prefix == 'get_https':
        out['get_https_status_code'] = code
        out['get_https_status_class'] = f'{code // 100}xx' if isinstance(code, int) else None
        headers = resp.get('headers') or {}
        out['get_https_server'] = firstval(headers.get('server'))
        out['get_https_content_type'] = firstval(headers.get('content_type'))
        out['get_https_header_count'] = len(headers)
        body = resp.get('body')
        out['get_https_body_len'] = len(body) if isinstance(body, str) else None
    else:
        out['get_http_status_code'] = code
        out['get_http_server'] = firstval((resp.get('headers') or {}).get('server'))
    return out

t0 = time.time()
get_store = {}
for tag, prefix, path in [
    ('openvpn', 'get_https', OUT / 'ipv4/OpenVPN/ipv4-openvpn-tcp.vpn.2022-08-01-1935-https.txt'),
    ('openvpn', 'get_http',  OUT / 'ipv4/OpenVPN/ipv4-openvpn-tcp.vpn.2022-08-01-1935-http.txt'),
    ('sstp',    'get_https', OUT / 'ipv4/SSTP/ipv4-sstp.vpn.2022-08-09-1007-https.txt'),
    ('sstp',    'get_http',  OUT / 'ipv4/SSTP/ipv4-sstp.vpn.2022-08-09-1007-http.txt'),
]:
    n = 0
    for e in stream_jsonl(path):
        ipi = ipv4_to_int(e['ip'])
        if ipi not in u3_set:
            continue
        cur = get_store.setdefault(ipi, {k: None for k in HTTPS_KEYS + HTTP_KEYS})
        feats = flatten_get(e, prefix)
        for k, v in feats.items():
            if cur.get(k) is None:
                cur[k] = v
        n += 1
    print(f'{tag} {prefix}: {n:,} rows scanned')
for ipi in list(u3_set):
    get_store.setdefault(ipi, {k: None for k in HTTPS_KEYS + HTTP_KEYS})
print(f'GET pass {time.time() - t0:.1f}s')

openvpn get_https: 482,956 rows scanned


openvpn get_http: 482,956 rows scanned


sstp get_https: 187,214 rows scanned


sstp get_http: 187,214 rows scanned


GET pass 28.0s


In [5]:
RDNS_KEYS = ['rdns_present', 'rdns_domain_count', 'rdns_domain_len', 'rdns_label_count',
             'rdns_tld', 'rdns_first_label', 'rdns_has_vpn', 'rdns_has_ip_literal']
rdns_store = {}
t0 = time.time()
for e in stream_jsonl(OUT / 'ipv4/rDNS/ipv4-all.shuf.rdns.noerror.out'):
    rev = e['name'].replace('.in-addr.arpa.', '')
    ipi = ipv4_to_int('.'.join(rev.split('.')[::-1]))
    if ipi not in u3_set:
        continue
    doms = [str(a.get('data', '')).rstrip('.') for a in ((e.get('data') or {}).get('answers') or [])]
    doms = [d for d in doms if d]
    if not doms:
        continue
    d0 = doms[0]
    labels = d0.split('.')
    rdns_store[ipi] = {
        'rdns_present': True,
        'rdns_domain_count': len(doms),
        'rdns_domain_len': len(d0),
        'rdns_label_count': len(labels),
        'rdns_tld': labels[-1].lower(),
        'rdns_first_label': labels[0].lower(),
        'rdns_has_vpn': any('vpn' in d.lower() for d in doms),
        'rdns_has_ip_literal': any(bool(IPTOKEN_RE.search(d)) for d in doms),
    }
print(f'rDNS pass {time.time() - t0:.1f}s | IPs with PTR: {len(rdns_store):,} / {len(U3):,}')

rDNS pass 19.4s | IPs with PTR: 326,075 / 641,039


In [6]:
# ASN + prefix length: longest-prefix sweep over the shipped ipasn snapshot
def load_ipasn(path):
    starts, ends, asns, plens = [], [], [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.startswith(';') or not line.strip():
                continue
            prefix, asn = line.rstrip().split('\t')[:2]
            ip, l = prefix.split('/')
            l = int(l)
            n = ipv4_to_int(ip)
            starts.append(n); ends.append(n + (1 << (32 - l)) - 1); asns.append(int(asn)); plens.append(l)
    starts = np.asarray(starts, np.int64); ends = np.asarray(ends, np.int64)
    asns = np.asarray(asns, np.int64); plens = np.asarray(plens, np.int8)
    order = np.lexsort((-ends, starts))
    return starts[order], ends[order], asns[order], plens[order]

v4_starts, v4_ends, v4_asns, v4_plens = load_ipasn(HIT / 'ipasn_ipv4_20220725.2200.dat')
asn_arr = np.zeros(len(U3), np.int64)
plen_arr = np.zeros(len(U3), np.int8)
asn_ok = np.zeros(len(U3), bool)
stack, i, n = [], 0, len(v4_starts)
for k, ip in enumerate(U3.tolist()):
    while i < n and v4_starts[i] <= ip:
        s = v4_starts[i]
        while stack and stack[-1][0] < s:
            stack.pop()
        stack.append((int(v4_ends[i]), int(v4_asns[i]), int(v4_plens[i])))
        i += 1
    while stack and stack[-1][0] < ip:
        stack.pop()
    if stack:
        asn_arr[k] = stack[-1][1]
        plen_arr[k] = stack[-1][2]
        asn_ok[k] = True
print(f'ASN resolved for {100 * asn_ok.mean():.2f}% of U3 ({(~asn_ok).sum():,} unmatched)')

ASN resolved for 99.90% of U3 (627 unmatched)


In [7]:
# assemble the per-IP table
tls_df = pd.DataFrame.from_dict(tls_store, orient='index').reindex(U3)
get_df = pd.DataFrame.from_dict(get_store, orient='index').reindex(U3)
rdns_df = pd.DataFrame.from_dict(rdns_store, orient='index').reindex(U3)
rdns_df['rdns_present'] = rdns_df['rdns_present'].fillna(False)

df = pd.DataFrame(index=U3)
df['ip'] = [int_to_ipv4(x) for x in U3]
df['ip_24'] = [int_to_ipv4(x & 0xFFFFFF00) for x in U3]
df['label'] = label
df['in_openvpn_tcp'] = in_ovpn
df['in_sstp'] = in_sstp
df['in_openvpn_udp'] = in_ovpn_udp
df['group_cert_fp'] = pd.Series(fp_by_ip).reindex(U3)
df = pd.concat([df, tls_df, get_df, rdns_df], axis=1)
df['asn'] = asn_arr
df['prefix_len'] = plen_arr
df.loc[~asn_ok, 'asn'] = None

# per-class coverage of the main feature families
cov = pd.DataFrame({
    'handshake_or_cert': df['handshake_present'].fillna(False) | df['cert_present'].fillna(False),
    'certificate': df['cert_present'].fillna(False),
    'get_https': df['get_https_response'].fillna(False),
    'rDNS': df['rdns_present'].fillna(False),
    'ASN': df['asn'].notna(),
})
cov['label'] = df['label']
display((100 * cov.groupby('label').mean()).round(1))
print('table shape:', df.shape)

del tls_store, get_store, rdns_store, tls_df, get_df, rdns_df, cov

,handshake_or_cert,certificate,get_https,rDNS,ASN
label,,,,,
both,98.6,98.637191,99.1,56.187567,100.0
openvpn,62.4,62.389247,59.6,53.020437,99.9
sstp,98.2,98.171846,75.7,43.702991,100.0


table shape: (641039, 65)


## 3. Freeze the split (before training)

Groups = *certificate fingerprint when present, else the /24 prefix*. A seeded greedy assignment
keeps the class proportions close to 70/20/10 while keeping every group intact. The three files plus
`split_metadata.json` (seed, counts, SHA-256 hashes) are written **before** any model is fitted; training then
reloads those exact files.

In [8]:
t0 = time.time()

# ---- groups = certificate fingerprint when present, else the /24 ----
group_key = np.where(df['group_cert_fp'].notna(),
                     'cert:' + df['group_cert_fp'].astype(str),
                     'net:' + df['ip_24'].astype(str))
df['group_id'] = group_key
uroots, inv = np.unique(group_key, return_inverse=True)
n_comp = len(uroots)
comp_size = np.bincount(inv).astype(float)
print(f'groups: {n_comp:,} | largest group: {int(comp_size.max()):,} rows '
      f'| singleton groups: {int((comp_size == 1).sum()):,}')

classes = ['openvpn', 'sstp', 'both']
y = df['label'].map({c: i for i, c in enumerate(classes)}).to_numpy()
comp_cls = np.stack([np.bincount(inv, weights=(y == c).astype(float), minlength=n_comp) for c in range(3)])
N_c = comp_cls.sum(axis=1)
frac = np.array([0.7, 0.2, 0.1])

rng = np.random.default_rng(SPLIT_SEED)
cur_cls = np.zeros((3, 3)); cur_tot = np.zeros(3)
assigned = np.full(n_comp, -1, np.int8)
for r in rng.permutation(n_comp):
    best_s, best_val = 0, np.inf
    for s in range(3):
        # minimize the CHANGE in cost (marginal greedy), not the current-state cost
        val = (float((cur_cls[s] - frac[s] * N_c) @ comp_cls[:, r])
               + 0.01 * (cur_tot[s] - frac[s] * len(U3)) * comp_size[r])
        if val < best_val:
            best_s, best_val = s, val
    assigned[r] = best_s
    cur_cls[best_s] += comp_cls[:, r]
    cur_tot[best_s] += comp_size[r]

split = assigned[inv]                     # 0=train, 1=test, 2=validation
df['split'] = split
print(f'components: {n_comp:,} | split {time.time() - t0:.1f}s')
display(pd.crosstab(df['label'], df['split']).rename(columns={0: 'train_70', 1: 'test_20', 2: 'validation_10'}))

chk = df.dropna(subset=['group_cert_fp']).groupby('group_cert_fp')['split'].nunique()
print('fingerprint groups spanning >1 split:', int((chk > 1).sum()))
assert (chk <= 1).all()
train_share = (df['split'] == 0).mean()
print(f'train share: {train_share:.3f} | test {((df["split"] == 1).mean()):.3f} | '
      f'validation {((df["split"] == 2).mean()):.3f}')
assert train_share > 0.6, 'greedy split failed to fill train; check group sizes'

groups: 327,741 | largest group: 10,684 rows | singleton groups: 273,327


components: 327,741 | split 3.2s


split,train_70,test_20,validation_10
label,,,
both,20392,5826,2913
openvpn,317677,90764,45384
sstp,110659,31616,15808


fingerprint groups spanning >1 split: 0
train share: 0.700 | test 0.200 | validation 0.100


In [9]:
SPLIT_DIR.mkdir(exist_ok=True)
names = {0: 'train_70.csv', 1: 'test_20.csv', 2: 'validation_10_unseen.csv'}
for s, fname in names.items():
    part = df[df['split'] == s].drop(columns=['split']).reset_index(drop=True)
    part.to_csv(SPLIT_DIR / fname, index=False)
    print(f'{fname:<28} rows={len(part):>8,}')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

meta = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'split_seed': SPLIT_SEED,
    'universe': 'H(openvpn_tcp) union H(sstp), IPv4',
    'n_rows': int(len(df)),
    'group_definition': 'certificate sha256 fingerprint when present, else the /24 prefix',
    'assignment': 'seeded greedy assignment to 70/20/10 minimizing class/size deviation',
    'class_counts': pd.crosstab(df['label'], df['split']).rename(columns={0: 'train_70', 1: 'test_20', 2: 'validation_10'}).to_dict(),
    'files': {fname: {'rows': int((df['split'] == s).sum()), 'sha256': sha256(SPLIT_DIR / fname)} for s, fname in names.items()},
    'columns': list(df.drop(columns=['split']).columns),
    'semi_leaky_columns': ['in_openvpn_udp'],
    'label_definition': "openvpn = in H(openvpn-tcp) and not in H(sstp); sstp = in H(sstp) and not in H(openvpn-tcp); both = in both",
    'leakage_rules': ['no protocol-probe responses used as features',
                      'features joined by IP only; no source-file provenance',
                      'in_openvpn_udp excluded from baseline models'],
}
with open(SPLIT_DIR / 'split_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print('wrote', SPLIT_DIR / 'split_metadata.json')

train_70.csv                 rows= 448,728


test_20.csv                  rows= 128,206


validation_10_unseen.csv     rows=  64,105


wrote /home/eric-farrall/vpnecosystem_dataset/tls_clf_splits/split_metadata.json


## 4. Models (trained only after the split was written)

Models reload the three frozen CSV files. Baselines: majority class and a hand rule
(Microsoft server header -> SSTP, self-signed/WatchGuard/Stormshield -> OpenVPN). Main model:
`HistGradientBoostingClassifier` with train-fitted top-K category codings (unseen categories -> "other",
missing -> its own code). Metrics on **test_20** and the final **validation_10_unseen**.

In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score

train = pd.read_csv(SPLIT_DIR / 'train_70.csv')
test = pd.read_csv(SPLIT_DIR / 'test_20.csv')
val = pd.read_csv(SPLIT_DIR / 'validation_10_unseen.csv')
classes = ['openvpn', 'sstp', 'both']
Y = {k: v['label'].map({c: i for i, c in enumerate(classes)}).to_numpy() for k, v in
     [('train', train), ('test', test), ('val', val)]}

DROP_ALWAYS = ['ip', 'ip_24', 'label', 'group_cert_fp', 'group_id', 'in_openvpn_tcp', 'in_sstp',
               'in_openvpn_udp', 'cert_subject_cn', 'cert_issuer_cn', 'asn']
CAT_K = {'tls_version': 16, 'tls_cipher': 24, 'cert_sig_algorithm': 24, 'cert_key_algorithm': 8,
         'cert_key_curve': 8, 'cert_validation_level': 8, 'cert_issuer_org': 64, 'cert_subject_org': 64,
         'get_https_probe_status': 8, 'get_https_status_class': 8, 'get_https_server': 64,
         'get_https_content_type': 24, 'get_http_probe_status': 8, 'get_http_server': 64,
         'rdns_tld': 64, 'rdns_first_label': 64}
NUM_COLS = ['cke_rsa_params_len', 'cert_validity_days', 'cert_age_days', 'cert_key_bits',
            'cert_san_dns_count', 'cert_names_count', 'get_https_body_len', 'get_https_header_count',
            'rdns_domain_count', 'rdns_domain_len', 'rdns_label_count', 'prefix_len']
BOOL_COLS = ['handshake_present', 'cert_present', 'tls_secure_renegotiation', 'tls_ticket',
             'tls_ocsp_stapling', 'tls_heartbeat_ext', 'tls_extended_master_secret', 'cke_rsa_present',
             'cert_expired', 'cert_self_signed', 'cert_sig_valid', 'cert_has_wildcard', 'cert_cn_is_ipv4',
             'cert_cn_has_softether', 'cert_cn_localhost', 'cert_subject_eq_issuer', 'cert_is_ca',
             'cert_key_encipherment', 'cert_digital_signature', 'cert_server_auth', 'get_https_response',
             'get_http_response', 'rdns_present', 'rdns_has_vpn', 'rdns_has_ip_literal']
ALL_FEATURES = [c for c in train.columns if c not in DROP_ALWAYS]

def build_xy(frame, cols, maps=None):
    X = pd.DataFrame(index=frame.index)
    new_maps = {}
    for c in cols:
        if c in CAT_K:
            s = frame[c].astype('object').where(frame[c].notna(), None)
            if maps is None:
                mapping = {v: i + 1 for i, v in enumerate(s.value_counts().head(CAT_K[c]).index)}
            else:
                mapping = maps[c]
            code = s.map(mapping)
            X[c] = np.where(s.isna(), 0, code.fillna(len(mapping) + 1)).astype('int32')
            new_maps[c] = mapping
        else:
            X[c] = pd.to_numeric(frame[c], errors='coerce').astype('float32')
    cat_idx = [X.columns.get_loc(c) for c in X.columns if c in CAT_K]
    return X, cat_idx, new_maps

def fit_eval(cols, max_iter=200, tag=''):
    Xtr, cat_idx, maps = build_xy(train, cols)
    Xte, _, _ = build_xy(test, cols, maps)
    Xva, _, _ = build_xy(val, cols, maps)
    clf = HistGradientBoostingClassifier(max_iter=max_iter, learning_rate=0.1, early_stopping=True,
                                         validation_fraction=0.1, n_iter_no_change=15,
                                         categorical_features=cat_idx, random_state=SPLIT_SEED)
    clf.fit(Xtr, Y['train'])
    out = {'model': tag or 'full', 'n_features': len(cols)}
    for name, X, y in [('test', Xte, Y['test']), ('validation', Xva, Y['val'])]:
        pred = clf.predict(X)
        out[f'{name}_acc'] = accuracy_score(y, pred)
        out[f'{name}_macroF1'] = f1_score(y, pred, average='macro')
        if name == 'validation':
            out['val_per_class_F1'] = dict(zip(classes, f1_score(y, pred, average=None).round(3)))
            out['val_confusion'] = confusion_matrix(y, pred).tolist()
    return out, clf

# majority + rule baselines on test
maj = Counter(Y['train']).most_common(1)[0][0]
rule_pred = np.where(test['get_https_server'].fillna('').str.contains('Microsoft', case=False),
                     classes.index('sstp'),
                     np.where(test['cert_self_signed'].fillna(False) |
                              test['cert_issuer_org'].isin(['WatchGuard', 'Stormshield']),
                              classes.index('openvpn'), classes.index('openvpn')))
print(f'majority baseline test acc : {accuracy_score(Y["test"], np.full(len(Y["test"]), maj)):.4f}')
print(f'rule baseline     test acc : {accuracy_score(Y["test"], rule_pred):.4f} '
      f'| macroF1 {f1_score(Y["test"], rule_pred, average="macro"):.4f}')

res, clf = fit_eval(ALL_FEATURES, max_iter=250, tag='full (3-class)')
print(json.dumps({k: v for k, v in res.items() if k != "val_confusion"}, indent=1, default=float))
print('validation confusion matrix [rows=true, cols=pred]:')
display(pd.DataFrame(res['val_confusion'], index=classes, columns=classes))

/tmp/ipykernel_30273/2560045979.py:4: DtypeWarning: Columns (0: cert_key_curve) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(SPLIT_DIR / 'train_70.csv')


majority baseline test acc : 0.7080
rule baseline     test acc : 0.8761 | macroF1 0.5808


{
 "model": "full (3-class)",
 "n_features": 55,
 "test_acc": 0.9814595260752227,
 "test_macroF1": 0.944822020706023,
 "validation_acc": 0.984478589813587,
 "validation_macroF1": 0.9561351145016653,
 "val_per_class_F1": {
  "openvpn": 0.993,
  "sstp": 0.976,
  "both": 0.899
 }
}
validation confusion matrix [rows=true, cols=pred]:


,openvpn,sstp,both
openvpn,45140,50,194
sstp,308,15140,360
both,69,14,2830


In [11]:
# feature-family ablations (test set, macro-F1) + binary OpenVPN-vs-SSTP variant
CERT_COLS = [c for c in ALL_FEATURES if c.startswith(('cert_', 'cke_')) or c == 'handshake_present']
GET_COLS = [c for c in ALL_FEATURES if c.startswith('get_')]
RDNS_COLS = [c for c in ALL_FEATURES if c.startswith('rdns_')]
var_sets = {
    'full': ALL_FEATURES,
    'no GET': [c for c in ALL_FEATURES if c not in GET_COLS],
    'no cert/TLS': [c for c in ALL_FEATURES if c not in CERT_COLS],
    'cert/TLS only': CERT_COLS,
    'GET only': GET_COLS,
    'rDNS only': RDNS_COLS,
}
rows = []
for tag, cols in var_sets.items():
    r, _ = fit_eval(cols, max_iter=150, tag=tag)
    rows.append({'variant': tag, 'n_features': r['n_features'],
                 'test_acc': round(r['test_acc'], 4), 'test_macroF1': round(r['test_macroF1'], 4)})
display(pd.DataFrame(rows).set_index('variant'))

binary = df[df['label'] != 'both']
btrain, btest, bval = (binary[binary['split'] == s].reset_index(drop=True) for s in (0, 1, 2))
yb = {k: v['label'].map({'openvpn': 0, 'sstp': 1}).to_numpy() for k, v in
      [('train', btrain), ('test', btest), ('val', bval)]}
Xtr, cat_idx, maps = build_xy(btrain, ALL_FEATURES)
Xte, _, _ = build_xy(btest, ALL_FEATURES, maps)
Xva, _, _ = build_xy(bval, ALL_FEATURES, maps)
bclf = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.1, early_stopping=True,
                                      validation_fraction=0.1, n_iter_no_change=15,
                                      categorical_features=cat_idx, random_state=SPLIT_SEED)
bclf.fit(Xtr, yb['train'])
for name, X, y in [('test', Xte, yb['test']), ('validation', Xva, yb['val'])]:
    p = bclf.predict(X)
    print(f'binary OpenVPN-vs-SSTP {name:<10}: acc {accuracy_score(y, p):.4f} | '
          f'macroF1 {f1_score(y, p, average="macro"):.4f} | ROC-AUC {roc_auc_score(y, bclf.predict_proba(X)[:, 1]):.4f}')

,n_features,test_acc,test_macroF1
variant,,,
full,55,0.9815,0.9448
no GET,43,0.9700,0.9257
no cert/TLS,28,0.9816,0.9439
cert/TLS only,27,0.9326,0.8894
GET only,12,0.9340,0.8831
rDNS only,8,0.7180,0.3533


binary OpenVPN-vs-SSTP test      : acc 0.9921 | macroF1 0.9896 | ROC-AUC 0.9994


binary OpenVPN-vs-SSTP validation: acc 0.9924 | macroF1 0.9901 | ROC-AUC 0.9989


## 5. Is this useful? Where the regime is real

**Realistic cases where you cannot run the protocol probe but can use these features:**

1. **Tooling.** The authors' detection needed *custom ZGrab2 VPN modules* (OpenVPN handshake construction,
   SSTP request format, PPTP parsing). TLS/HTTP scanning (`zgrab2 tls`, `curl`) and rDNS/ASN are commodity or
   passive. If you cannot deploy/maintain protocol-specific probes, fingerprint attribution is the only option.
2. **Policy / ethics.** Many operators allow standard TLS/HTTP measurement but not emulated VPN-protocol
   handshakes (which resemble attack traffic).
3. **Passive-only vantage points.** An ISP or monitor may hold TLS/HTTP metadata (certificates, SNI,
   server headers) or DNS/rDNS and flow records but cannot emit probes at all. The model works on features
   derivable from that telemetry.
4. **Gaps in protocol probing.** The PAM method cannot detect OpenVPN with `tls-auth`/unknown keys,
   PSK protocols (WireGuard, AnyConnect), or VPNs on non-registered ports. A TLS/HTTP/rDNS fingerprint can
   still attribute such servers *once they are candidates* from another source.
5. **Triangulation and monitoring.** Independent cross-checks of active-measurement hitlists, co-hosting
   detection (the `both` class), and cheap periodic re-fingerprinting of known fleets.

**Honest caveats.**

* It assumes you already have a **candidate set** of VPN-ish :443 servers; it does not discover them.
  In deployment the candidate pool would also contain millions of ordinary HTTPS hosts, whose false-positive
  behavior is untested here.
* Training labels come from the very probes this model replaces, so servers the probes miss are absent
  (selection bias).
* Much of the signal is **vendor/software fingerprint** (Microsoft HTTP stack, WatchGuard/softether certs):
  expect drift when vendors/versions change, and do not read it as protocol semantics.
* Reachability is usually not the blocker -- if you can TLS-scan a host you can often also probe it; the real
  asymmetry is tooling, policy, and passivity. ASN alone carries almost no protocol signal (measured earlier);
  rDNS alone is weak. The model's value is concentrated in certificate + HTTP fingerprints on :443.

In [12]:
print('Frozen split files:')
for fname in names.values():
    p = SPLIT_DIR / fname
    print(f'  {p}  ({p.stat().st_size / 1e6:.1f} MB)')
print('Metadata:', SPLIT_DIR / 'split_metadata.json')

Frozen split files:
  /home/eric-farrall/vpnecosystem_dataset/tls_clf_splits/train_70.csv  (213.5 MB)
  /home/eric-farrall/vpnecosystem_dataset/tls_clf_splits/test_20.csv  (61.3 MB)
  /home/eric-farrall/vpnecosystem_dataset/tls_clf_splits/validation_10_unseen.csv  (29.9 MB)
Metadata: /home/eric-farrall/vpnecosystem_dataset/tls_clf_splits/split_metadata.json
